# Risk-Aware Bike Routing: Routing Prototype (Nov checkpoint)

Builds Manhattan's bike street network as a graph (intersections = nodes, street segments = edges),
then compares two routes for the same trip:

- **Shortest distance**: Dijkstra on segment length
- **Bike-lane preferred**: same algorithm, but streets without a bike lane cost 1.5x their length

The edge-cost function already has a placeholder `risk` term; that's where the team's street-level risk score plugs in later.

In [ ]:
# First run only (in a terminal): pip install -r requirements.txt
import os, re
import networkx as nx
import osmnx as ox
import folium

print("osmnx", ox.__version__)

## 1. Build the street graph

In [ ]:
PLACE = "Manhattan, New York, USA"
GRAPH_FILE = "manhattan_bike.graphml"

# Keep the OpenStreetMap tags that say whether a street has a bike lane
ox.settings.useful_tags_way = list(dict.fromkeys(list(ox.settings.useful_tags_way) + [
    "cycleway", "cycleway:left", "cycleway:right", "cycleway:both", "bicycle"]))

if os.path.exists(GRAPH_FILE):
    G = ox.load_graphml(GRAPH_FILE)
else:
    G = ox.graph_from_place(PLACE, network_type="bike")  # a minute or two the first time
    ox.save_graphml(G, GRAPH_FILE)                       # cached for next time

print(f"Intersections (nodes):   {G.number_of_nodes():,}")
print(f"Street segments (edges): {G.number_of_edges():,}")

## 2. Mark bike lanes and define the edge cost

In [ ]:
BIKE_LANE_VALUES = {"cycleway", "lane", "track", "opposite_lane", "opposite_track"}
TAG_KEYS = ["highway", "cycleway", "cycleway:left", "cycleway:right", "cycleway:both"]

def has_bike_lane(data):
    """True if the OSM tags say this segment is a bike path or has a painted/protected lane."""
    text = " ".join(str(data.get(k, "")) for k in TAG_KEYS)
    return bool(BIKE_LANE_VALUES & set(re.findall(r"[a-z_]+", text)))

def edge_cost(data, off_lane_penalty=1.5, risk_weight=0.0):
    """Cost of riding one street segment.

    off_lane_penalty: how much longer a street with no bike lane "counts"
    risk_weight (lambda): how much the risk score matters.
    risk is a placeholder (0) until the team's risk model is ready.
    """
    penalty = 1.0 if data["bike_lane"] else off_lane_penalty
    risk = data.get("risk", 0.0)
    return data["length"] * penalty * (1 + risk_weight * risk)

for u, v, k, data in G.edges(keys=True, data=True):
    data["length"] = float(data["length"])
    data["bike_lane"] = has_bike_lane(data)
    data["bike_cost"] = edge_cost(data)

n_lane = sum(d["bike_lane"] for _, _, d in G.edges(data=True))
print(f"{n_lane / G.number_of_edges():.0%} of street segments have a bike lane")

## 3. Pick a trip

In [ ]:
# (latitude, longitude) -- change these to try other trips
ORIGIN = (40.7359, -73.9911)       # Union Square
DESTINATION = (40.7681, -73.9819)  # Columbus Circle

orig = ox.distance.nearest_nodes(G, X=ORIGIN[1], Y=ORIGIN[0])
dest = ox.distance.nearest_nodes(G, X=DESTINATION[1], Y=DESTINATION[0])

## 4. Compute both routes (Dijkstra)

In [ ]:
def route_edges(G, route, weight):
    """Edge data along a route (picks the cheapest of any parallel edges)."""
    return [min(G[u][v].values(), key=lambda d: d[weight]) for u, v in zip(route[:-1], route[1:])]

def route_stats(G, route, weight):
    edges = route_edges(G, route, weight)
    total = sum(d["length"] for d in edges)
    on_lane = sum(d["length"] for d in edges if d["bike_lane"])
    return {"km": total / 1000,
            "pct_bike_lane": 100 * on_lane / total,
            "minutes": total / 1000 / 15 * 60}  # assumes ~15 km/h riding speed

# nx.shortest_path with a weight runs Dijkstra's algorithm
routes = {
    "Shortest distance":   (nx.shortest_path(G, orig, dest, weight="length"),    "length"),
    "Bike-lane preferred": (nx.shortest_path(G, orig, dest, weight="bike_cost"), "bike_cost"),
}

base_km = None
for name, (route, w) in routes.items():
    s = route_stats(G, route, w)
    extra = "" if base_km is None else f"   (+{100 * (s['km'] / base_km - 1):.0f}% distance)"
    base_km = base_km or s["km"]
    print(f"{name:20s} {s['km']:.2f} km   ~{s['minutes']:.0f} min   "
          f"{s['pct_bike_lane']:.0f}% on bike lanes{extra}")

## 5. Map both routes

In [ ]:
def route_coords(G, route, weight):
    """(lat, lon) points along a route, following the street shape where OSM has it."""
    pts = []
    for (u, v), d in zip(zip(route[:-1], route[1:]), route_edges(G, route, weight)):
        if "geometry" in d:
            pts.extend((lat, lon) for lon, lat in d["geometry"].coords)
        else:
            pts.extend([(G.nodes[u]["y"], G.nodes[u]["x"]), (G.nodes[v]["y"], G.nodes[v]["x"])])
    return pts

m = folium.Map(location=ORIGIN, zoom_start=14)
style = {"Shortest distance": ("#d62728", 8), "Bike-lane preferred": ("#2ca02c", 4)}
for name, (route, w) in routes.items():
    s = route_stats(G, route, w)
    color, width = style[name]
    folium.PolyLine(route_coords(G, route, w), color=color, weight=width, opacity=0.85,
                    tooltip=f"{name}: {s['km']:.2f} km, {s['pct_bike_lane']:.0f}% bike lanes").add_to(m)
folium.Marker(ORIGIN, tooltip="Start", icon=folium.Icon(color="blue")).add_to(m)
folium.Marker(DESTINATION, tooltip="End", icon=folium.Icon(color="black")).add_to(m)

m.save("routes_map.html")
print("Saved routes_map.html -- open it in a browser (red = shortest, green = bike-lane preferred)")

## Next steps

- Replace the placeholder `risk` with the team's street-level risk score (an edge attribute called `risk`),
  then vary `risk_weight` to show the safety vs. travel-time tradeoff
- Use the NYC Open Data bike-route layer instead of OSM tags once it's joined to the street segments
- Try A* (`nx.astar_path`) with a straight-line-distance heuristic
- Interface for choosing origin and destination on the map